# SQL Retail Sales & Profitability Analysis

This project uses SQL to analyze retail sales data and identify key drivers of revenue and profitability.

The analysis focuses on category performance, discount strategy, customer segments, shipping modes, and regional performance. It also demonstrates intermediate to advanced SQL concepts, including joins, common table expressions (CTEs), case statements, and window functions.

## Objective

The goal of this project is to answer business-focused questions using SQL and to demonstrate practical querying skills relevant to data analyst roles.

This notebook uses the Sample Superstore dataset and a SQLite database created inside Jupyter for fast, reproducible analysis.

## Business Questions

1. Which product categories generate the most sales and profit?
2. Which sub-categories are the most and least profitable?
3. How do discounts affect profitability?
4. Which customer segments contribute the most value?
5. How do shipping modes differ in usage and profitability?
6. Which states underperform?
7. Which patterns become clearer through ranking, contribution analysis, and cumulative totals?

## Import SQLite

In [1]:
import pandas as pd
import sqlite3

## Load Dataset

The dataset is loaded from CSV and prepared for SQL analysis

In [2]:
df = pd.read_csv("SampleSuperstore.csv", encoding="latin1")
df.head()

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [3]:
df.columns.tolist()

['Ship Mode',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Category',
 'Sub-Category',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

## Clean the columns for SQL use

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns.tolist()

['ship_mode',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'category',
 'sub_category',
 'sales',
 'quantity',
 'discount',
 'profit']

# Create SQLite Database

A SQLite database is created inside the notebook. Although the original dataset is a single flat table, it is split into smaller logical tables to demonstrate SQL joins in a more realistic way.

In [5]:
conn = sqlite3.connect("superstore.db")                          # Create a local SQL databse called 'superstore.db'
df.to_sql("orders", conn, if_exists="replace", index=False)      # Write the dataframe into a table called 'orders'

9994

In [6]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)               # Verify it worked

,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,sales,quantity,discount,profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


# Category performance

In [14]:
q1 = """
SELECT
    category,
    ROUND(SUM(sales), 2) AS total_sales,      -- SUM(sales) total revenue
    ROUND(SUM(profit), 2) AS total_profit     -- SUM(profit) total profit
FROM orders
GROUP BY category                             -- group data by category 
ORDER BY total_sales DESC;                    -- order by, ranking 
"""

pd.read_sql(q1, conn)

,category,total_sales,total_profit
0,Technology,836154.03,145454.95
1,Furniture,741999.80,18451.27
2,Office Supplies,719047.03,122490.80


#### Insight

Technology generates the highest sales and profit, indicating strong overall performance.

#### Business Implication

Revenue and profitability are aligned for Technology, making it a key driver of business value.

# Sub-category profitability

In [16]:
q2 = """
SELECT
    sub_category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM orders
GROUP BY sub_category                         -- group by sub-category 
ORDER BY total_profit ASC;                    -- Ascending order 
"""

pd.read_sql(q2, conn)

,sub_category,total_sales,total_profit
0,Tables,206965.53,-17725.48
1,Bookcases,114880.00,-3472.56
2,Supplies,46673.54,-1189.10
3,Fasteners,3024.28,949.52
4,Machines,189238.63,3384.76
5,Labels,12486.31,5546.25
6,Art,27118.79,6527.79
7,Envelopes,16476.40,6964.18
8,Furnishings,91705.16,13059.14
9,Appliances,107532.16,18138.01


#### Insight

While some sub-categories such as Phones and Copiers generate strong profits, others such as Tables, Bookcases, and Supplies show negative total profit.

#### Business Implication

This indicates that overall category performance can hide underperforming sub-categories. Loss-making products should be reviewed for pricing, discounting, or cost inefficiencies.

# Discount vs. Profit

In [19]:
q3 = """
SELECT
    discount,
    ROUND(AVG(profit), 2) AS avg_profit,
    COUNT(*) AS num_orders
FROM orders
GROUP BY discount
ORDER BY discount;
"""

pd.read_sql(q3, conn)

,discount,avg_profit,num_orders
0,0.00,66.90,4798
1,0.10,96.06,94
2,0.15,27.29,52
3,0.20,24.70,3657
4,0.30,-45.68,227
5,0.32,-88.56,27
6,0.40,-111.93,206
7,0.45,-226.65,11
8,0.50,-310.70,66
9,0.60,-43.08,138


#### Insight

There is a clear negative relationship between discount levels and profitability. Lower discount levels are associated with positive average profit, while higher discounts significantly reduce or even reverse profitability. With highest profit at 10% discount, and highest loss at 50% discount. 

#### Business Implication

Aggressive discounting appears to be a major driver of losses. Pricing strategies should be carefully evaluated to avoid margin erosion, especially for already underperforming sub-categories.

# Customer segment analysis

In [20]:
q4 = """
SELECT
    segment,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit), 2) AS avg_profit_per_order,
    COUNT(*) AS order_count
FROM orders
GROUP BY segment
ORDER BY total_sales DESC;
"""

pd.read_sql(q4, conn)

,segment,total_sales,total_profit,avg_profit_per_order,order_count
0,Consumer,1161401.34,134119.21,25.84,5191
1,Corporate,706146.37,91979.13,30.46,3020
2,Home Office,429653.15,60298.68,33.82,1783


#### Insight

The Consumer segment drives the highest total sales and profit, indicating it is the primary revenue source. However, differences in average profit per order suggest that not all segments are equally efficient. Home Office segment, followed by Corporate segment, has highest average profit per order. 

#### Business Implication

While the Consumer segment is the largest contributor, evaluating profitability per order can help identify more efficient customer segments and inform targeting strategies.

# Shipping mode analysis

In [21]:
q5 = """
SELECT
    ship_mode,
    COUNT(*) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit), 2) AS avg_profit_per_order
FROM orders
GROUP BY ship_mode
ORDER BY total_sales DESC;
"""

pd.read_sql(q5, conn)

,ship_mode,order_count,total_sales,total_profit,avg_profit_per_order
0,Standard Class,5968,1358215.74,164088.79,27.49
1,Second Class,1945,459193.57,57446.64,29.54
2,First Class,1538,351428.42,48969.84,31.84
3,Same Day,543,128363.13,15891.76,29.27


#### Insight

Standard Class dominates in terms of order volume and total sales. However, other shipping modes show higher average profit per order, indicating differences in efficiency.

#### Business Implication

While Standard shipping supports scale, faster or premium shipping options may generate higher value per transaction. Balancing volume and efficiency could improve overall profitability.

# State-level performance

In [22]:
q6 = """
SELECT
    state,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM orders
GROUP BY state
ORDER BY total_profit ASC
LIMIT 15;
"""

pd.read_sql(q6, conn)

,state,total_sales,total_profit
0,Texas,170188.05,-25729.36
1,Ohio,78258.14,-16971.38
2,Pennsylvania,116511.91,-15559.96
3,Illinois,80166.10,-12607.89
4,North Carolina,55603.16,-7490.91
5,Colorado,32108.12,-6527.86
6,Tennessee,30661.87,-5341.69
7,Arizona,35282.00,-3427.92
8,Florida,89473.71,-3399.30
9,Oregon,17431.15,-1190.47


#### Insight

Several states such as Texas, Ohio, Pennsylvania, show negative total profit despite generating sales, indicating regional inefficiencies.

#### Business Implication

Geographic performance varies significantly. Underperforming states should be analyzed further for pricing, cost structure, or demand differences to improve profitability.

# Discount bands (grouped)  using CASE WHEN

In [23]:
q7 = """
SELECT
    CASE
        WHEN discount = 0 THEN 'No Discount'
        WHEN discount > 0 AND discount <= 0.2 THEN 'Low Discount'
        WHEN discount > 0.2 AND discount <= 0.4 THEN 'Medium Discount'
        ELSE 'High Discount'
    END AS discount_band,
    COUNT(*) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit), 2) AS avg_profit
FROM orders
GROUP BY discount_band
ORDER BY total_profit DESC;
"""

pd.read_sql(q7, conn)

,discount_band,order_count,total_sales,total_profit,avg_profit
0,No Discount,4798,1087908.47,320987.60,66.90
1,Low Discount,3803,846522.24,100785.47,26.50
2,Medium Discount,460,234137.90,-35817.47,-77.86
3,High Discount,933,128632.25,-99558.59,-106.71


#### Insight

Grouping discounts into broad bands makes the profitability pattern easier to interpret.

Low or no discount orders generate the strongest profit, while heavily discounted orders contribute disproportionately low profit or losses.

#### Business Implication

Discount strategy appears to be a major driver of profitability. Broad discounting should be controlled more carefully, especially for categories and sub-categories that already underperform.

# CTE for loss-making sub-categories

In [26]:
q8 = """
WITH subcategory_profit AS (                          -- WITH....AS creates temporary result set with category, sub-category, total sales, total profit
    SELECT
        category,
        sub_category,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders
    GROUP BY category, sub_category
)
SELECT
    category,
    sub_category,
    total_sales,
    total_profit
FROM subcategory_profit
WHERE total_profit < 0                               -- Filter the results to show only sub-categories with total negative profit
ORDER BY total_profit ASC;
"""

pd.read_sql(q8, conn)

,category,sub_category,total_sales,total_profit
0,Furniture,Tables,206965.53,-17725.48
1,Furniture,Bookcases,114880.00,-3472.56
2,Office Supplies,Supplies,46673.54,-1189.10


#### Insight

The CTE isolates sub-categories with negative total profit, making it easier to identify persistent loss drivers.

This confirms that underperformance is concentrated in specific product groups (Tables, Bookcases, and Supplies) rather than spread evenly across the business.

#### Business Implication

Targeted action on loss-making sub-categories could improve profitability more effectively than broad, category-wide changes.

# Window Function — Rank sub-categories within each category

In [28]:
q9 = """
SELECT
    category,
    sub_category,
    total_profit,
    RANK() OVER (                                    -- RANK() OVER () to rank within each category, sub-categories from most to least profitable
        PARTITION BY category
        ORDER BY total_profit DESC
    ) AS profit_rank_within_category
FROM (
    SELECT
        category,
        sub_category,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders
    GROUP BY category, sub_category
) t
ORDER BY category, profit_rank_within_category;
"""

pd.read_sql(q9, conn)

,category,sub_category,total_profit,profit_rank_within_category
0,Furniture,Chairs,26590.17,1
1,Furniture,Furnishings,13059.14,2
2,Furniture,Bookcases,-3472.56,3
3,Furniture,Tables,-17725.48,4
4,Office Supplies,Paper,34053.57,1
5,Office Supplies,Binders,30221.76,2
6,Office Supplies,Storage,21278.83,3
7,Office Supplies,Appliances,18138.01,4
8,Office Supplies,Envelopes,6964.18,5
9,Office Supplies,Art,6527.79,6


### Insight

Ranking sub-categories within each category reveals which product groups drive the strongest and weakest performance relative to their peers.

This makes it easier to compare performance inside each category rather than across the business as a whole.

### Business Implication

Window functions such as `RANK()` support prioritization by showing which sub-categories should be protected, promoted, or reviewed within each product category.

# Percentage contribution of each category to total sales

In [30]:
q10 = """
SELECT
    category,
    ROUND(total_sales, 2) AS total_sales,
    ROUND(
        100.0 * total_sales / SUM(total_sales) OVER (),          -- each category’s percentage share of total sales -- SUM() OVER () calculates total sales across all categories 
        2
    ) AS pct_of_total_sales
FROM (
    SELECT
        category,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY category
) t
ORDER BY total_sales DESC;
"""

pd.read_sql(q10, conn)

,category,total_sales,pct_of_total_sales
0,Technology,836154.03,36.4
1,Furniture,741999.80,32.3
2,Office Supplies,719047.03,31.3


#### Insight

This query shows how much each category contributes to total sales across the business.

Technology contributes the largest share of revenue, reinforcing its importance as a key sales driver.

#### Business Implication

Percentage contribution provides context beyond raw sales totals and helps prioritize categories based on their share of overall business performance.

# Top product in each category by sales using ROW_NUMBER()

In [31]:
q11 = """
SELECT
    category,
    sub_category,
    total_sales
FROM (
    SELECT
        category,
        sub_category,
        ROUND(SUM(sales), 2) AS total_sales,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY SUM(sales) DESC
        ) AS rn
    FROM orders
    GROUP BY category, sub_category
) t
WHERE rn = 1
ORDER BY total_sales DESC;
"""

pd.read_sql(q11, conn)

,category,sub_category,total_sales
0,Technology,Phones,330007.05
1,Furniture,Chairs,328449.10
2,Office Supplies,Storage,223843.61


#### Insight

This query identifies the top-performing sub-category within each product category by sales.

It highlights which sub-categories drive category-level revenue and helps distinguish category leaders from weaker contributors.

#### Business Implication

This analysis can support product prioritization, inventory planning, and category-level decision-making by showing which sub-categories contribute the most revenue within each category.

# Key Findings

- Technology is the strongest category in both sales and profitability
- Several sub-categories, including Tables, Bookcases, and Supplies, generate negative profit
- Higher discounts are associated with lower average profit, indicating margin erosion
- The Consumer segment contributes the highest total sales and profit
- Standard shipping dominates order volume, while other shipping modes may generate higher average profit per order
- Window functions help reveal ranking and contribution patterns that are not obvious from simple aggregations alone

# Conclusion

This SQL analysis shows that profitability is influenced more by discount strategy, product mix, and sub-category performance than by revenue alone.

The results suggest that the business could improve profitability by controlling aggressive discounting, reviewing loss-making sub-categories, and focusing on the strongest-performing product areas.

This project demonstrates practical SQL skills relevant to data analyst roles, including aggregation, filtering, CASE statements, CTEs, and window functions.

# Technical Notes

- Database: SQLite
- Environment: Jupyter Notebook
- Data preparation: Python and Pandas used only to load the CSV and create the SQL table
- SQL concepts demonstrated: GROUP BY, ORDER BY, CASE WHEN, CTEs, RANK(), ROW_NUMBER(), and window-based percentage calculations

In [32]:
conn.close()